# Appendix: score-guided balancing

This notebook compares regressor balancing and covariate balancing. It is motivated by the Neyman-score view: ATE under heterogeneous treatment effects generally requires balancing functions of the full regressor `X=(D,Z)`, whereas ATT counterfactual mean estimation can be well aligned with covariate balancing on `Z`. The current high-level `grr_att` wrapper targets the full ATT effect functional, so the covariate-only ATT counterfactual-mean variant is not forced into this notebook; ATT rows use the regressor-balancing basis supported by the wrapper.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from inside the genriesz repository.")
    REPO_ROOT = parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz import grr_ate, grr_att
from genriesz.basis import BaseBasis, TreatmentInteractionBasis
from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    CoverageDiagnosticBasis,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    generator_shift_for_estimand,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_compatible_generator,
    make_coverage_diagnostic_data,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"
TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_LABELS = {
    "ra": "RA",
    "rw": "RW (IPW)",
    "arw": "ARW (AIPW)",
    "tmle": "TMLE",
    "SQ": "SQ-Riesz",
    "UKL": "UKL-Riesz",
    "BKL": "BKL-Riesz",
    "BP(0.5)": "BP-Riesz (omega = 0.5)",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random forest leaves",
    "rff": "Random Fourier features",
    "matching": "Nearest-neighbor matching",
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")
pd.options.display.max_rows = TABLE_CONFIG["max_rows"]


def label_of(value):
    """Return the display label for a stored result key."""

    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    """Replace stored result keys inside a composite display label."""

    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def prettify_labels(frame):
    """Return a copy with known result-key columns formatted for display."""

    out = frame.copy()
    for column in LABEL_COLUMNS:
        if column in out.columns:
            out[column] = out[column].map(label_of)
    return out


def display_table(frame, *, caption=None, digits=4):
    """Display a rounded table without changing the stored results."""

    table = prettify_labels(frame)
    numeric_columns = table.select_dtypes(include=[np.number]).columns
    table[numeric_columns] = table[numeric_columns].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)


In [ ]:
N_REPLICATIONS = 100
SAMPLE_SIZE = 1200
BASIS_KIND = "rkhs"
RIEZ_FEATURES = 120
RIEZ_LAMBDA = 3e-1
BALANCE_MODES = ["regressor", "covariate"]
# This appendix isolates the choice of functions to balance.  We use UKL-Riesz
# as the fixed branchwise compatible loss to avoid the trivial alpha=0 solution
# that an unconstrained SQ loss can produce with covariate-only ATE features.
# Loss-family comparisons are handled in the main and model-variation notebooks.
LOSS_SPECS_SCORE = [
    {"loss": "UKL", "omega": None, "label": "UKL"},
]
FOLDS = 5

In [ ]:
rows = []
for heterogeneous in [False, True]:
    for rep in range(N_REPLICATIONS):
        data = make_score_guided_data(n=SAMPLE_SIZE, seed=700000 + 1009 * rep + int(heterogeneous), heterogeneous=heterogeneous)
        for basis_mode in ["covariate", "regressor"]:
            for loss_spec in LOSS_SPECS_SCORE:
                for estimand in ESTIMANDS:
                    # A covariate-only basis makes m_basis_matrix identically zero for the
                    # package's full ATT-effect wrapper.  Estimating the ATT counterfactual
                    # mean by pure covariate balancing would require a separate functional,
                    # so we skip that unsupported combination instead of plotting a degenerate result.
                    if estimand == "ATT" and basis_mode == "covariate":
                        continue
                    fit_rows = fit_one_grr(
                        data,
                        estimand=estimand,
                        loss_spec=loss_spec,
                        basis_kind=BASIS_KIND,
                        basis_mode=basis_mode,
                        outcome_basis_kind=BASIS_KIND,
                        outcome_basis_mode="regressor",
                        cross_fit=True,
                        lam=RIEZ_LAMBDA,
                        basis_features=RIEZ_FEATURES,
                        folds=FOLDS,
                        estimators=ESTIMATORS_ALL,
                        random_state=rep,
                    )
                    for row in fit_rows:
                        row["heterogeneous_effect"] = bool(heterogeneous)
                        row["replication"] = rep
                    rows.extend(fit_rows)
score_balance_results = pd.DataFrame(rows)
score_balance_summary = summarize_estimates(score_balance_results, ["heterogeneous_effect", "estimand", "basis_mode", "loss", "estimator"])

for estimand_name in ESTIMANDS:
    table_df = score_balance_summary[score_balance_summary["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["heterogeneous_effect", "basis_mode", "loss", "estimator"])
    display_table(table_df, caption=f"Appendix score-guided balancing: {estimand_name}")


In [ ]:
# Box plots comparing regressor and covariate balancing. ATE and ATT are separated.
plot_df = score_balance_results[(score_balance_results["status"] == "ok") & (score_balance_results["estimator"].isin(["rw", "arw"]))].copy()
plot_df["method"] = plot_df["basis_mode"] + " | " + plot_df["estimator"]
plot_df["squared_error_plot"] = plot_df["squared_error"].clip(lower=PLOT_CONFIG["squared_error_floor"])

for estimand_name in ESTIMANDS:
    for heterogeneous in [False, True]:
        panel_df = plot_df[(plot_df["estimand"] == estimand_name) & (plot_df["heterogeneous_effect"] == heterogeneous)].copy()
        if panel_df.empty:
            print(f"No plot data for {estimand_name}, heterogeneous={heterogeneous}.")
            continue
        fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size_wide"], dpi=PLOT_CONFIG["dpi"])
        ordered_methods = sorted(panel_df["method"].unique())
        box_data = [panel_df.loc[panel_df["method"] == m, "squared_error_plot"].dropna().to_numpy() for m in ordered_methods]
        box = ax.boxplot(box_data, tick_labels=[prettify_method(_m) for _m in ordered_methods], widths=PLOT_CONFIG["box_width"], patch_artist=True, showfliers=False)
        for patch, method in zip(box["boxes"], ordered_methods):
            mode = method.split(" | ")[0]
            patch.set_facecolor("#4C78A8" if mode == "regressor" else "#F58518")
            patch.set_alpha(0.75)
        ax.set_yscale(PLOT_CONFIG["squared_error_y_scale"])
        ax.set_title(f"{estimand_name}: score-guided balance, heterogeneous={heterogeneous}", fontsize=PLOT_CONFIG["title_fontsize"])
        ax.set_xlabel("Balance mode and estimator", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.set_ylabel("Squared error", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.tick_params(axis="x", labelrotation=45, labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
        fig.tight_layout()
        plt.show()
